# Explaining the network with exact Shapley values

**Explanation notebook 1 in `notebooks/xai/`.** `notebooks/README.md` says how every notebook is laid out and who it is written for. `notebooks/xai/README.md` says what every explanation notebook shares — the same trained network, the same engine, the same stand-in for a removed sensor — and why.

## What this notebook is for

Notebook 02 trained a network that reads 14 sensors' last 30 flight cycles and predicts how many cycles of life the engine has left. This notebook takes one of those predictions at a time and divides it among the 14 sensors, using the method Wen et al. (2026) use: the **Shapley value**. It then repeats the division at every cycle of one engine's life and draws the result, so you can see which sensors the prediction leaned on while the engine was healthy, and which it leaned on as the engine wore out. Last, it counts how often the four largest contributions change from one window to the next, on this project's default network and on one with the paper's recurrent stage, beside the paper's own Fig. 10.

## What it does not do

- **It does not train anything.** It loads a network that `scripts/train.py` trained and stored with the commit of the code that trained it.
- **It does not say what wears an engine.** A Shapley value describes how this network used its inputs. Section 7 lists what else it does not show.
- **Its numbers are not the paper's Table 12.** That table has the same layout — sensors down the side, moments in one engine's life across the top — but it cannot hold signed Shapley values that add up the way the paper's own Equation 14 requires (finding F-08). The two are never set side by side as the same quantity: step 8 never compares the values themselves, only how often the four largest change and how far each sensor's value moves from one window to the next relative to its own size.

## The notebook in one list

| # | Step | What goes in | What comes out |
|---|------|--------------|----------------|
| 1 | **Load** the trained network | a stored five-seed run | seed 42's network, and the data rebuilt as that network saw it |
| 2 | **Choose** what removing a sensor means | the training windows | one stand-in window, and the network's prediction for it |
| 3 | **Work three sensors by hand** | one window, three sensors | all 6 orders they could join in, and their 3 Shapley values |
| 4 | **Compute all fourteen**, exactly | the same window | 14 Shapley values that add up to the prediction minus the stand-in's |
| 5 | **Follow one engine's life** | every window of one held-back engine | 14 values at every cycle, drawn as 14 small charts |
| 6 | **Rank the sensors**, overall and by stage of life | those values | which sensors mattered, and how that changed as the engine wore |
| 7 | **Check the other four networks** | the run's 5 networks and the 100 test engines | one ranking per network, and how closely they agree |
| 8 | **Count how often the four largest change** | two other stored runs, alike but for the temporal stage, five seeds each, and the paper's Table 12 | how often a sensor enters the four largest from one window to the next, the paper's Fig. 10 redrawn beside this project's engine, and a paired test |

## 1. Background

### What exactly is being explained

One **prediction**. The network receives one example, an array of shape `(14, 30)` — 14 sensors, each with its readings over the 30 most recent flight cycles — and returns one number: the predicted remaining life, in flight cycles. The question is how much of that number each of the 14 sensors is responsible for.

### Why that question has no obvious answer

If the network added up fourteen separate terms, one per sensor, each term would be that sensor's share and there would be nothing to discuss. It does not. Its graph layers mix every sensor with the sensors it is joined to, and after each layer every negative number is set to zero, so how far one sensor moves the prediction depends on what the other sensors are reading.

A made-up function of two inputs shows the difficulty:

```
f(a, b) = 2a + 3b + 4ab
```

Compare the input `a = 1, b = 1` with a reference input `a = 0, b = 0`. The whole change to explain is `f(1, 1) − f(0, 0) = 9 − 0 = 9`. How much of the 9 belongs to `a`?

- Switch `a` on first, with `b` still at 0: `f(1, 0) − f(0, 0) = 2 − 0 = 2`. Then switch `b` on: `f(1, 1) − f(1, 0) = 9 − 2 = 7`.
- Switch `b` on first: `f(0, 1) − f(0, 0) = 3 − 0 = 3`. Then switch `a` on: `f(1, 1) − f(0, 1) = 9 − 3 = 6`.

`a` adds 2 or 6 depending on whether `b` is already there. The reason is the `4ab` term, an **interaction**: a part of the output that neither input produces on its own. Neither 2 nor 6 is wrong. Each is the right answer for one order of arrival.

### The Shapley value: average over every order

The **Shapley value** (Shapley, 1953) settles the question by averaging each player's contribution over every order in which the players could arrive. It comes from cooperative game theory, the mathematics of how a group that produced something together should divide the credit. Here the players are sensors, and what they produced together is the prediction.

In the example, `a`'s Shapley value is `(2 + 6) / 2 = 4` and `b`'s is `(7 + 3) / 2 = 5`. They add up to 9, the whole change, and the interaction's 4 has been split evenly between the two inputs that make it. The adding up is not luck. In each order, the amounts added along the way sum to the change from nothing switched on to everything switched on — that is simply how the steps were defined — so the averages over the orders sum to it too. This property is called **efficiency**, and step 4 checks it.

### What "switching a sensor off" means for a network

In the example, "off" meant 0. A network built for fourteen sensors cannot be run with thirteen, so a sensor is never really removed. Its 30 readings are **replaced** by a stand-in. This notebook's stand-in is that sensor's average window over the training engines: "this sensor is behaving typically" (assumption A-20). Step 2 builds it. The prediction for a window made entirely of stand-ins is the **reference prediction**. It plays the part of "nothing switched on" in the example, and every Shapley value below is measured from it.

**Do not confuse** a Shapley value with an attention weight or a gradient. A Shapley value is obtained by changing the inputs and running the network again, it is in the unit of the prediction (here, cycles), and the fourteen values of one window add up to a real difference between two predictions. The network's parameters are never read.

## 2. Setup

The first cell imports what the notebook uses and makes the package importable from a clone that has not been installed. The second finds the stored run to explain: the most recent five-seed run of `config/fd001.yaml` with the paper's sensor graph, the last-cycle temporal stage, the network sizes of the paper's Table 4 and the step learning-rate schedule, whose trained networks were saved. If there is none, it stops and prints the command that makes one.

In [ ]:
from __future__ import annotations

import hashlib
import json
import logging
import sys
import time
from dataclasses import replace
from itertools import permutations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from matplotlib import patheffects
from matplotlib.colors import to_rgba
from matplotlib.patches import Circle
from scipy import stats

# The project's top folder is the nearest one above this notebook that holds
# pyproject.toml. Adding its src/ folder to Python's search path makes the
# package importable from a clone that was never installed.
PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    import torch
except ImportError as missing:
    raise ImportError(
        "This notebook runs a trained network, which needs PyTorch, and PyTorch is "
        'not installed. Install the project with  pip install -e ".[model]"'
    ) from missing

from thesis_gnn_rul_xai.config import load_config
from thesis_gnn_rul_xai.docsite.figures import render_figure
from thesis_gnn_rul_xai.evaluation.published import (
    FIG_10_RING,
    SHAP_TABLE_12,
    SHAP_TABLE_12_GRAPHS,
)
from thesis_gnn_rul_xai.explain import (
    coalition_weights,
    dynamic_shap,
    efficiency_error,
    mean_window_baseline,
    plot_contributions_over_time,
    shap_for_each_seed,
    shap_summary,
    shap_values,
)
from thesis_gnn_rul_xai.graph.adjacency import build_physics_adjacency
from thesis_gnn_rul_xai.models import ModelConfig
from thesis_gnn_rul_xai.training import evaluate_model, find_runs, load_seed, read_run
from thesis_gnn_rul_xai.utils.logging_setup import configure_logging
from thesis_gnn_rul_xai.utils.provenance import current_commit

configure_logging(logging.ERROR)
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

print(f"Project root: {PROJECT_ROOT}")
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")

In [ ]:
config = load_config(PROJECT_ROOT / "config" / "fd001.yaml")
runs = [
    run for run in find_runs(
        preprocessing=config, seeds=5, with_weights=True, split="varies_with_seed",
        early_stopping=True, lr_schedule="step", temporal_stage="last_cycle",
    )
    if run.model_config == ModelConfig()
]
if not runs:
    raise FileNotFoundError(
        "reports/ holds no five-seed run of config/fd001.yaml that saved its trained "
        "networks. Make one, which takes a few minutes on a processor, with\n\n"
        "    python scripts/train.py --config config/fd001.yaml --seeds 5\n\n"
        "and run this notebook again."
    )
run = runs[-1]
SEED = 42

print(f"Stored run : {run.name}")
print(f"Commit     : {run.summary['commit']}")
print(f"Finished   : {run.summary['finished_utc']} (UTC)")
print(f"Seeds      : {run.seeds}. Steps 1 to 6 explain seed {SEED}; step 7 checks all five.")

## 3. The trained network

### Step 1 — Load the trained network

**What.** Load the parameters seed 42's network ended its training with, and rebuild the data it was trained on from the settings stored beside them.

**Why.** An explanation has to be an explanation of something definite. Loading the stored network, rather than training one here, means the network explained is the one whose test error is part of the reported five-seed result, and the same one every other notebook in `xai/` explains. The data has to be rebuilt with the same seed, because the seed decided which 20 training engines were held back: rebuilding with another seed would call engines "unseen" that this network learned from.

In [ ]:
model, prepared = load_seed(run, SEED)
sensors = list(prepared.sensor_symbols)

print(f"Network: {model.count_parameters():,} parameters; sensor nodes {', '.join(sensors)}")
print(f"Data   : {len(prepared.train):,} training windows, "
      f"{len(prepared.validation):,} held-back windows, {len(prepared.test)} test windows")
print(f"One example has shape {prepared.test.features.shape[1:]}: 14 sensors, 30 cycles each.")

**Check.** If the loaded network is the stored one, and the data is the data it saw, then scoring the 100 test engines again must give exactly the error the run recorded for seed 42.

In [ ]:
recorded = run.results.set_index("seed").loc[SEED]
scored = evaluate_model(model, prepared.test)
print(f"test RMSE, scored now        {scored['rmse']:.6f} cycles")
print(f"test RMSE, recorded by run   {recorded['rmse']:.6f} cycles")

same_network = bool(np.isclose(scored["rmse"], recorded["rmse"], rtol=0, atol=1e-6))
assert same_network, "The loaded network does not score what the stored run recorded."
print("\nOK  The network explained below is the one the stored run trained.")

## 4. The explanation, step by step

### Step 2 — The stand-in for a removed sensor

**What.** Average every training window, sensor by sensor and cycle by cycle, into one window of shape `(14, 30)`, and run the network on it.

**Why.** Equation 15 of the paper needs a prediction "when only sensors in subset S are considered", and the paper never says what the network is given in place of the others. This notebook gives it each missing sensor's average training window (assumption A-20). Only training windows go into the average, so no held-back or test engine shapes the explanation, for the same reason none of them shapes the rescaling.

Notebook 01 rescaled every sensor to count how many spreads (standard deviations) a reading sits from that sensor's training average. So the stand-in should be close to zero everywhere: the average window is, almost, "every reading exactly average". It is not exactly zero, because windows count the middle cycles of an engine's life more often than its first and last ones.

In [ ]:
baseline = mean_window_baseline(prepared.train)
with torch.no_grad():
    reference = float(model.predict_cycles(torch.from_numpy(baseline[None]))[0])

print(f"stand-in window: shape {baseline.shape}, averaged over {len(prepared.train):,} training windows")
print(f"its entry furthest from zero: {np.abs(baseline).max():.3f} spreads")
print(f"reference prediction, for a window made entirely of stand-ins: {reference:.3f} cycles")

**Check.** The reference prediction is a prediction like any other, so it must lie between 0 and the ceiling of 125 cycles, the most this network's last layer can produce.

In [ ]:
reference_in_range = 0.0 <= reference <= config.rul_max
assert reference_in_range
print(f"OK  0 <= {reference:.3f} <= {config.rul_max}")

### Step 3 — Three sensors, worked by hand

**What.** Take one real window: the last window of the held-back engine that step 5 follows, a moment close to its failure. Pick three sensors, `Nc`, `T30` and `P15`. Keep the other eleven at their real readings the whole time, and treat the three as the only players. With three players there are 2 × 2 × 2 = 8 groups of them that could be switched on, and 3 × 2 × 1 = 6 orders in which they could arrive. That is few enough to list every one.

**Why.** Equation 15 is this same calculation with fourteen players. With three, every number can be checked by eye. This is a smaller game than the one step 4 plays — eleven sensors are never switched off here — so these three values are not what the same sensors receive in step 4. They show how the calculation works, not what the sensors are worth.

The engine is chosen once, here, and used again in step 5: the held-back engine with the most windows. It is one the network never learned from, and it has the longest recorded life of the held-back engines, so it shows the most of an engine's wear.

The figure lists every order three players, A, B and C, could arrive in. B's Shapley value is what B adds on arriving, averaged over all six rows.

In [ ]:
display(HTML(render_figure("shapley")))

In [ ]:
held_back = prepared.validation
units, counts = np.unique(held_back.units, return_counts=True)
engine = int(units[np.argmax(counts)])
engine_rows = np.flatnonzero(held_back.units == engine)
engine_rows = engine_rows[np.argsort(held_back.end_cycles[engine_rows])]

last = engine_rows[-1]
window = held_back.features[last]
players = ["Nc", "T30", "P15"]
position = {sensor: sensors.index(sensor) for sensor in players}


def prediction_with(switched_on) -> float:
    """The prediction when only these players keep their real readings."""
    example = window.copy()
    for sensor in players:
        if sensor not in switched_on:
            example[position[sensor]] = baseline[position[sensor]]
    with torch.no_grad():
        return float(model.predict_cycles(torch.from_numpy(example[None]))[0])


groups = [(), ("Nc",), ("T30",), ("P15",), ("Nc", "T30"), ("Nc", "P15"),
          ("T30", "P15"), ("Nc", "T30", "P15")]
value = {frozenset(group): prediction_with(group) for group in groups}

print(f"Engine {engine}: {len(engine_rows)} windows. The one used here ends at cycle "
      f"{held_back.end_cycles[last]}, with {held_back.targets[last]:.0f} cycles truly left.")
print(f"The network predicts {value[frozenset(players)]:.3f} cycles for it.\n")
display(pd.DataFrame({
    "switched on": [", ".join(group) or "none of the three" for group in groups],
    "prediction (cycles)": [value[frozenset(group)] for group in groups],
}).round(3))

The table is everything the calculation needs: eight predictions. The first row, none of the three switched on, is not the reference prediction of step 2, because the other eleven sensors keep their real readings in this smaller game.

The next cell goes through the six orders. Along each order it records what each sensor adds on arriving: the prediction after it joins, minus the prediction before. The column averages are the Shapley values of this three-sensor game.

In [ ]:
records = []
for order in permutations(players):
    switched_on, added = [], {}
    for sensor in order:
        before = value[frozenset(switched_on)]
        switched_on.append(sensor)
        added[sensor] = value[frozenset(switched_on)] - before
    records.append({"order of arrival": " then ".join(order),
                    **{f"{sensor} adds": added[sensor] for sensor in players}})

orders = pd.DataFrame(records)
display(orders.round(3))

by_orders = {sensor: float(orders[f"{sensor} adds"].mean()) for sensor in players}
total_change = value[frozenset(players)] - value[frozenset()]
print("Averaged over the six orders:")
for sensor in players:
    print(f"  {sensor:4s} {by_orders[sensor]:+9.3f} cycles")
print(f"  sum  {sum(by_orders.values()):+9.3f}")
print(f"\nprediction with all three  minus  prediction with none: {total_change:+9.3f}")

Look down `Nc`'s column. When `Nc` arrives first it moves the prediction by −8.606 cycles; when it arrives after `T30` it moves it by only −5.196. Once `T30` has already pulled the prediction down, `Nc` has less left to add: the two sensors partly tell the network the same thing about this engine. That is an interaction, like the `4ab` term of section 1, and it is why no single row can be called `Nc`'s contribution. The Shapley value, −6.835, is the average of all six.

**Check.** Efficiency: the three values must add up to the whole change, the prediction with all three switched on minus the prediction with none.

The same numbers can also be reached without listing orders, and that is the form Equation 15 uses. Count the orders in which `Nc` arrives to find exactly a given group already there. `Nc` arrives first, finding nobody, in 2 of the 6 orders (a share of 1/3). It arrives second, after `T30` alone, in 1 of the 6 (1/6), and likewise after `P15` alone (1/6). It arrives last, finding both, in 2 of the 6 (1/3). So `Nc`'s value is those four shares multiplied by what `Nc` adds to each group, summed. `coalition_weights(3)` returns the shares by group size, 1/3, 1/6 and 1/3, and the cell checks that the two routes agree.

In [ ]:
three_add_up = abs(sum(by_orders.values()) - total_change) < 1e-6
assert three_add_up
print(f"OK  {sum(by_orders.values()):+.6f} = {total_change:+.6f}")

weights = coalition_weights(3)
print(f"\nShares by group size, from coalition_weights(3): {np.round(weights, 4).tolist()}")
by_groups = {
    sensor: sum(
        weights[len(group)] * (value[frozenset(group) | {sensor}] - value[frozenset(group)])
        for group in groups if sensor not in group
    )
    for sensor in players
}
routes_agree = all(abs(by_groups[s] - by_orders[s]) < 1e-6 for s in players)
assert routes_agree
for sensor in players:
    print(f"  {sensor:4s} by orders {by_orders[sensor]:+9.3f}   by groups {by_groups[sensor]:+9.3f}")
print("OK  Averaging over orders and weighting groups give the same values.")

### Step 4 — All fourteen sensors, exactly

**What.** The same window, now with every sensor a player. This is the paper's Equation 15:

$$\phi_i \;=\; \sum_{S \,\subseteq\, N \setminus \{i\}} \frac{|S|!\,(n-|S|-1)!}{n!}\,\Big(f(S \cup \{i\}) - f(S)\Big)$$

- $n = 14$, the number of sensors, and $N$ is the set of all fourteen.
- $i$ is the sensor being credited, and $\phi_i$ its Shapley value, in cycles.
- $S$ is a group of the other sensors — any group, from none of them to all thirteen — and $|S|$ is how many sensors it holds.
- $f(S)$ is the prediction when the sensors in $S$ keep their real readings and every other sensor is replaced by its stand-in.
- $f(S \cup \{i\}) - f(S)$ is what sensor $i$ adds to the group $S$ — one entry of step 3's order table.
- The fraction is the share of all $n!$ orders in which exactly the sensors of $S$ arrive before $i$: the same count step 3 did by hand for three sensors, where it gave 1/3, 1/6 and 1/3.

The whole sum is step 3's average over every order, computed group by group instead of order by order.

**Why it can be exact here.** Fourteen sensors can arrive in 14! = 87,178,291,200 orders, but those orders only ever pass through 2¹⁴ = 16,384 distinct groups, and each group needs one run of the network. `shap_values` runs the network on all 16,384 versions of the window and applies the weights. Most software for Shapley values has to sample orders instead, because with many more inputs the groups cannot all be listed. Nothing is sampled here, so a disagreement with the paper can never be blamed on sampling noise.

In [ ]:
phi = shap_values(model, window, baseline)
with torch.no_grad():
    predicted = float(model.predict_cycles(torch.from_numpy(window[None]))[0])

table = pd.DataFrame({"sensor": sensors, "Shapley value (cycles)": phi})
table = table.reindex(table["Shapley value (cycles)"].abs().sort_values(ascending=False).index)
display(table.round(3).reset_index(drop=True))

print(f"prediction                          {predicted:9.3f} cycles")
print(f"reference prediction                {reference:9.3f}")
print(f"sum of the 14 Shapley values        {phi.sum():+9.3f}")
print(f"prediction minus reference          {predicted - reference:+9.3f}")
print()
for sensor in players:
    print(f"{sensor:4s} in the three-sensor game {by_orders[sensor]:+8.3f}   among all fourteen {phi[position[sensor]]:+8.3f}")

The last three lines show why step 3's values were only an illustration. In the three-sensor game the other eleven sensors were always at their real readings. Among all fourteen, each sensor's contribution is averaged over groups in which those eleven are often replaced by stand-ins, and because of interactions that changes what it adds.

**Check.** Efficiency, which is the paper's Equation 14. The paper writes that equation with a constant it calls the average value of the target variable across all samples. The only constant these values can add up to is the reference prediction, so that is what they are measured from (assumption A-24). If the sum missed by more than rounding, the numbers would not be Shapley values however sensible they looked.

In [ ]:
window_error = efficiency_error(model, window, baseline, phi)
fourteen_add_up = window_error < 1e-4
assert fourteen_add_up
print(f"OK  The 14 values miss (prediction minus reference) by {window_error:.1e} cycles.")

### Step 5 — Every cycle of one engine's life

**What.** Step 4, repeated at every window of the engine chosen in step 3: the window ending at cycle 30, the one ending at cycle 31, and so on to the engine's last recorded cycle. This is the paper's Equation 16, which is Equation 15 applied at each time step.

**Why.** One prediction shows what the network leaned on at one moment. The paper's claim — and the reason its title calls its explanation *dynamic* — is that what matters changes as an engine degrades. Following one engine through its whole life shows that directly. A held-back engine is used because the network never learned from it; a test engine cannot be, because the test set keeps only one window per engine.

In [ ]:
started = time.perf_counter()
attribution = dynamic_shap(model, held_back, baseline, unit=engine, symbols=sensors)
seconds = time.perf_counter() - started

print(f"Engine {engine}: {len(attribution)} windows x {2**14:,} groups = "
      f"{len(attribution) * 2**14:,} runs of the network, in {seconds:.1f} s.")
display(attribution.head(3).round(2))

Each row is one window: the cycle it ends at; the window's label, the number the network is trained to predict, which is the remaining life there capped at 125 cycles (assumption A-01), so it reads 125 wherever more than 125 cycles truly remain; the prediction; the reference prediction (the same in every row); and the fourteen Shapley values.

(Corrected 2026-10-03: this called the second column the true remaining life, and the description of the figure below called its grey line the same. They are the capped label: the check below finds it below the life truly left in 114 of engine 5's 240 windows. Found by a review of the wording around the capped label.)

**Check.** Efficiency at every one of the windows, not only the one of step 4; and that the label column is the remaining life capped at 125, counted against the life the engine truly had at each window. A held-back engine comes from the training file, which records every engine until it fails, so the life truly left at a window is the number of cycles from the window's last cycle to the engine's last.

In [ ]:
engine_gap = float(np.abs(attribution[sensors].sum(axis=1)
                          - (attribution["predicted"] - attribution["baseline"])).max())
every_window_adds_up = engine_gap < 1e-4
assert every_window_adds_up
print(f"OK  Across all {len(attribution)} windows the largest miss is {engine_gap:.1e} cycles.")

# The label column against the life truly left: a training engine runs to failure, so
# its last window ends at the cycle it failed.
truly_left = attribution["end_cycle"].max() - attribution["end_cycle"]
label_is_capped_life = bool((attribution["rul"] == np.minimum(truly_left, config.rul_max)).all())
assert label_is_capped_life
below_true_life = int((attribution["rul"] < truly_left).sum())
print(f"OK  The label is the remaining life capped at {config.rul_max}. The first window ends "
      f"at cycle {attribution['end_cycle'].iloc[0]} with {truly_left.iloc[0]} cycles truly left "
      f"and a label of {attribution['rul'].iloc[0]:.0f}; the label is below the life truly "
      f"left in {below_true_life} of {len(attribution)} windows.")

#### How each sensor's share moved over the engine's life

The figure has two parts.

- **The top panel** is what is being explained. The grey line is each window's label, the remaining life capped at 125 cycles (assumption A-01), which the legend calls "label: remaining life, capped": flat at the ceiling of 125 while more than 125 cycles remain, then falling one cycle per cycle to zero at failure. Where it is flat the engine truly had more life left than the line shows. The blue line is the network's prediction at each window. The dark horizontal line is the reference prediction, and the vertical line marks the cycle where the label first drops below the ceiling.
- **The fourteen small panels** are one sensor each: its Shapley value at every window, in cycles. A blue area means that sensor's readings pushed the prediction *up*, towards more life left, compared with typical readings. A red area means they pushed it *down*. The panels are ordered by average size, largest first, and all fourteen share one vertical scale, so a tall panel really is a large contribution.

At any single cycle, the fourteen panel heights added together (blue counted as positive, red as negative) give the distance between the blue line and the dark reference line in the top panel. That is efficiency again, drawn.

In [ ]:
figure = plot_contributions_over_time(
    attribution, sensors, title=f"Engine {engine}: the prediction, and where it came from"
)
plt.show()

#### Reading the figure

**While the engine is healthy, many sensors contribute and they pull in opposite directions.** `T30`, `htBleed`, `T24`, `T50` and `Ps30` push the prediction up, while `W31`, `P30`, `phi` and `W32` push it down, each by amounts of the same order. The prediction sits well above the reference because the pushes up outweigh the pushes down. Remember what "up" is measured against: the stand-in is the average training window, and the training windows include worn engines. A healthy engine's perfectly normal readings therefore count as "better than typical", and that is what the blue areas record.

**As the engine wears, the picture reorganises around one sensor.** `Nc`, the core speed, contributes almost nothing for the first half of the engine's life and then becomes by far the largest single contribution, pulling the prediction down. At the same time, the sensors that pushed up while the engine was healthy cross zero and push down, and several that pushed down cross to pushing up. Step 6 puts numbers on both movements.

**`P15` is nearly silent**, apart from narrow spikes early in the engine's life. It takes only two distinct values in the whole FD001 training set, so its readings change only when it flips from one to the other (finding F-01).

### Step 6 — Which sensors mattered, overall and by stage of life

**What.** Two summaries of step 5's table.

- **Overall**, from `shap_summary`: each sensor's average contribution size ignoring direction (`mean_abs`), its average signed contribution (`mean`, where a negative value means it pulled the prediction down on average), and the largest single contribution it made (`max_abs`).
- **By stage**: the average size of each sensor's contribution in the first, middle and last third of the engine's windows.

**Why.** The figure shows the shapes; these tables put numbers to them, and the stage table measures the change over the engine's life that the paper's dynamic explanation is about. The paper's Table 12 picks five consecutive moments at each of three stages of one engine. Thirds of this engine's life are used here instead, because this engine has a different length, and because averaging over a whole third is steadier than five moments.

In [ ]:
ranking = shap_summary(attribution[sensors].to_numpy(), sensors)
print(f"Engine {engine}, all {len(attribution)} windows, sorted by average size:")
display(ranking.round(2))

thirds = np.array_split(np.arange(len(attribution)), 3)
stages = pd.DataFrame({
    f"{name} third (cycles {attribution['end_cycle'].iloc[rows[0]]}-{attribution['end_cycle'].iloc[rows[-1]]})":
        attribution[sensors].iloc[rows].abs().mean()
    for name, rows in zip(("first", "middle", "last"), thirds)
})
stages["last minus first"] = stages.iloc[:, 2] - stages.iloc[:, 0]
print("Average size of each sensor's contribution, by third of the engine's life:")
display(stages.sort_values(stages.columns[2], ascending=False).round(2))

In the stage table, `Nc` goes from the smallest contribution of all fourteen in the first third to by far the largest in the last, and the last column shows it gaining more than any other sensor. `P30`, `phi` and `W31` lose the most. Near failure, then, this network's prediction rests largely on one sensor, where earlier it rested on many small pushes that partly cancelled.

**Check.** `P15` carries almost no information (finding F-01), so a sensible attribution should rank it last overall. It is not a proof that the attribution is right, but a ranking that put `P15` near the top would be a warning.

In [ ]:
p15_rank = int(ranking.index[ranking["sensor"] == "P15"][0]) + 1
p15_last = p15_rank == len(sensors)
assert p15_last
print(f"OK  P15 ranks {p15_rank} of {len(sensors)} for engine {engine}.")

late_leader = stages.iloc[:, 2].idxmax()
print(f"Largest average contribution in the last third: {late_leader}")

## 5. Is the ranking a property of the architecture, or of one network?

### Step 7 — The other four networks

**What.** Load each of the run's five networks, compute Shapley values for the last window of each of the 100 test engines, and rank the sensors by average contribution size for each network. Then measure how closely the five rankings agree.

**Why.** Everything above describes seed 42's network. The other four seeds trained networks of the same design on slightly different data (each held back different engines) from different starting values. A ranking that changes when only the seed changes describes that one network, not the design. The test engines are used here because they are the same 100 for every network, while the held-back engines differ from seed to seed. Explaining them decides nothing about any network, so it does not use up the test set.

**How agreement is measured.** Spearman's rank correlation compares two rankings: 1 means the same order, 0 no relationship, −1 the reverse order. With no ties it is `1 − 6 × Σd² / (m × (m² − 1))`, where `m` is the number of items ranked and `d` is how far apart one item's two ranks are. For three items ranked `[1, 2, 3]` by one network and `[1, 3, 2]` by another, the differences are `0, −1, 1`, so `Σd² = 2` and the correlation is `1 − 6 × 2 / (3 × 8) = 0.5`. Here `m = 14` sensors.

In [ ]:
sizes = {}
for seed in run.seeds:
    network, data = load_seed(run, seed)
    started = time.perf_counter()
    values = shap_values(network, data.test.features, mean_window_baseline(data.train))
    sizes[seed] = pd.Series(np.abs(values).mean(axis=0), index=sensors)
    print(f"seed {seed}: {len(data.test)} test windows in {time.perf_counter() - started:.1f} s; "
          f"largest four {sizes[seed].sort_values(ascending=False).index[:4].tolist()}")

sizes = pd.DataFrame(sizes)
ranks = sizes.rank(ascending=False).astype(int)
print("\nRank of each sensor by average contribution size, one column per network:")
display(ranks.sort_values(SEED))

agreement = sizes.corr(method="spearman")
off_diagonal = agreement.to_numpy()[~np.eye(len(agreement), dtype=bool)]
print("Spearman's rank correlation between every pair of networks:")
display(agreement.round(2))
print(f"lowest {off_diagonal.min():.2f}, highest {off_diagonal.max():.2f}")

**Check.** Two statements this notebook relies on must hold for every one of the five networks, not only seed 42's: `P15` is ranked last, and `Nc` and `T24` are the two sensors with the largest average contributions.

In [ ]:
p15_last_everywhere = bool((ranks.loc["P15"] == len(sensors)).all())
top_two_everywhere = all(
    set(sizes[seed].sort_values(ascending=False).index[:2]) == {"Nc", "T24"}
    for seed in run.seeds
)
assert p15_last_everywhere
assert top_two_everywhere
print("OK  Every network ranks P15 last.")
print("OK  Every network's two largest average contributions are Nc and T24, in one order or the other.")

The rankings agree closely, and the two statements hold for all five networks. So, for this design and these test engines, which sensors carry the largest contributions is not an accident of one seed. The correlation table shows one network, seed 44, agreeing a little less with the others than they do with each other, mostly because it ranks `T50` twelfth where the other four rank it between fourth and sixth.

## 6. How often the four largest contributions change

### Step 8 — The default network beside a recurrent one, and beside the paper's Fig. 10

**What the paper shows.** The paper's Fig. 10 draws fifteen small graphs of one engine, one per time step: graphs 1 to 5 early, 78 to 82 in the middle and 187 to 191 late. Which window of the engine a time step is, the paper does not say; the reading used here is set out below the next cell. In each, the four sensors with the largest Shapley values are coloured red; call them the **red four**. The paper's Table 12 prints the numbers behind the graphs, and the figure follows the table in 13 of the 15 (finding F-17). From one graph to the next inside a stage of five, the red four hardly changes.

**What this step measures.** How often the red four changes for this project's networks, on five seeds of two stored runs that differ only in the **temporal stage**, the first part of the network, which turns each sensor's 30 readings into what that sensor's graph node carries:

- the **default network** (`last_cycle`) hands the graph layers each sensor's most recent reading, as preprocessing left it after smoothing (assumptions A-07 and A-04);
- the **recurrent network** (`lstm`) runs each sensor's 30 readings through that sensor's own small recurrent network, which carries a summary forward from one cycle to the next, and hands over one number computed from all 30. That is the stage of the paper's Section 2.2, at a size the paper does not give (assumption A-26).

Neither is the paper's network, so nothing here is a reproduction of Fig. 10; the paper's numbers are set beside them to show what the paper's own explanation looked like.

**Why.** The playground's copy of Fig. 10 shows the default network's red four changing almost every window. Before a change of the red four is read as something that happened to the engine, a reader needs to know how often it changes on a network for reasons of the network's own.

The paper does not name its engine, and the engine explained here is another one, so what is compared with Table 12 is how often its red four changes, never which sensors are in it.

In [ ]:
# Two stored runs, alike in everything but the temporal stage. They are named
# rather than searched for, because the comparison needs the pair, made at one
# commit, and a later run of either kind would break the pair.
STAGE_RUNS = {
    "last_cycle": "fd001_physics_row_cap_5seed_varsplit_cfgba82e3_f386291-dirty_20260914T150004Z",
    "lstm": "fd001_physics_row_cap_5seed_varsplit_lstm32_cfg72f506_f386291-dirty_20260914T134955Z",
}
STAGE_NAMES = {"last_cycle": "default network", "lstm": "recurrent network"}
missing = [name for name in STAGE_RUNS.values()
           if not (PROJECT_ROOT / "reports" / name / "summary.json").is_file()]
if missing:
    raise FileNotFoundError(
        f"Step 8 explains two stored runs that are not in reports/: {missing}. Stored runs "
        "are not committed. A new pair, made at one commit, comes from\n\n"
        "    python scripts/train.py --config config/fd001.yaml --seeds 5\n"
        "    python scripts/train.py --config config/fd001.yaml --seeds 5 --temporal lstm\n\n"
        "and their names go in STAGE_RUNS above."
    )
stage_runs = {stage: read_run(PROJECT_ROOT / "reports" / name) for stage, name in STAGE_RUNS.items()}


def settings_apart_from_the_stage(stored):
    """Everything a run recorded about how it was made, except its temporal stage."""
    return (
        {k: v for k, v in stored.preprocessing.to_dict().items() if k != "notes"},
        {k: v for k, v in stored.summary["model"].items() if k != "temporal"},
        stored.summary["training"], stored.summary["split"], stored.summary["seeds"],
        stored.summary["reported_weights"], stored.summary["commit"],
    )


same_settings = (settings_apart_from_the_stage(stage_runs["last_cycle"])
                 == settings_apart_from_the_stage(stage_runs["lstm"]))
assert same_settings, "The two runs differ in more than their temporal stage."

def digest(preprocessing) -> str:
    """The six-character code scripts/train.py wrote into a run's folder name until 2026-10-03, notes included."""
    return hashlib.sha256(json.dumps(preprocessing.to_dict(), sort_keys=True, default=str).encode()).hexdigest()[:6]


digests = {stage: digest(stored.preprocessing) for stage, stored in stage_runs.items()}
assert all(f"_cfg{digests[stage]}_" in stored.name for stage, stored in stage_runs.items())
swapped_description = digest(replace(stage_runs["lstm"].preprocessing, notes=stage_runs["last_cycle"].preprocessing.notes))
digest_is_the_description = swapped_description == digests["last_cycle"]
assert digest_is_the_description, "The two runs' preprocessing settings differ in more than their description."

for stage, stored in stage_runs.items():
    print(f"{STAGE_NAMES[stage]:18s} {stored.name}")
    print(f"{'':18s} temporal stage {stage}; finished {stored.summary['finished_utc']} (UTC); "
          f"test RMSE over its 5 seeds {stored.summary['mean']['rmse']:.3f} ± "
          f"{stored.summary['std']['rmse']:.3f} cycles")
made = stage_runs["lstm"].summary
print(f"\nBoth trained at commit {made['commit']}, and alike in everything else: {made['subset']}, "
      f"the {made['graph']} graph rescaled by {made['adjacency_normalization']}, test labels capped: "
      f"{made['clip_test_labels']}, {made['validation_fraction']:.0%} of training engines held back "
      f"({made['split']}), {made['lr_schedule']} learning-rate schedule, weights kept: "
      f"{made['reported_weights']}, sizes {made['model']['gcn_hidden']} and {made['model']['fc_hidden']}.")
training = made["training"]
passes = {stage: stored.results["epochs_run"].tolist() for stage, stored in stage_runs.items()}
rate_dropped = max(max(p) for p in passes.values()) > training["epochs"] // 2
print(f"Training: at most {training['epochs']} passes, stopped early after {training['early_stopping_patience']} "
      f"passes without a lower validation error; passes run per seed: {passes}. The step schedule drops the "
      f"learning rate after pass {training['epochs'] // 2}, which {'some seed' if rate_dropped else 'no seed'} went past.")
print(f"Configuration digests {digests}; the recurrent run's preprocessing settings with the default "
      f"run's description in place of its own: {swapped_description}.")
print(f"Explained now with the code at commit {current_commit(PROJECT_ROOT)}.")

**What the cell above checked.** The two runs recorded the same settings in everything but the temporal stage. Their folder names still carry different configuration digests, `cfgba82e3` and `cfg72f506`. A configuration digest is a six-character code that `scripts/train.py` computes from the recorded preprocessing settings alone; the temporal stage and the training settings are not part of it, so a change of stage cannot change it. When these two runs were named, the code also counted a one-sentence description stored with the settings, and the two codes differ because the recurrent run, started from the playground, stored a different sentence: the cell puts the default run's sentence in its place and gets `ba82e3` back. Since 2026-10-03 the description is no longer counted, so both runs' settings now give `818b40`, the code the table in `src/thesis_gnn_rul_xai/training/README.md` lists beside `ba82e3` and `72f506`; the folder names keep the codes they were given. The function `digest` in the cell above follows the rule from before that date, the one that named these two folders, which is why it gives their codes back. Both name the commit `f386291-dirty`. The `-dirty` means the working tree held changes not yet committed when each run started, and the two finished about seventy minutes apart, so the shared name shows that they started from the same commit, not that every file was identical.

**How they were trained.** Both ran for at most 200 passes over the training windows with early stopping: training stopped after 20 passes in a row without a lower validation error, and the weights kept are those of the pass with the lowest. Every seed of both runs stopped between passes 28 and 75, so the step schedule's drop of the learning rate, due after pass 100, never happened, and every pass used the first learning rate, 0.001, the starting rate of the paper's Table 6.

**The commit of the explanations.** The last line printed above is the commit of the code that computed the explanations below, and it carries `-dirty` for the same reason: when this notebook was last executed, on 2026-10-03, the working tree held changes not yet committed. The code this step uses — `shap_for_each_seed` in `explain/seeds.py`, and the shortcut in `explain/shap.py` that runs the temporal stage once per window — is committed in `6733d77`. The changes were the renamed legend of step 5's figure, in `explain/plots.py`, which this step does not draw, and work in progress elsewhere in the project, which did not touch either of those two files. The line names the commit of the latest execution, and the figures below came out the same in every execution that day: at `19dceb5-dirty`, before that code was committed, from a clean tree at `6733d77`, which printed `6733d77`, and at `6733d77-dirty`.

**Which window is graph *g*.** Table 12 heads its fifteen columns "Time", and the paper's dynamic explanation "operates on each time step" (its Section 2.4, in the paragraph introducing Equation 16). The paper does not say which window of the engine a time step is. This notebook, like the playground, reads graph *g* as the engine's *g*-th window: graph 1 is the window ending at the engine's 30th cycle, graph 191 the one ending at its 220th. That reading is a choice, written up in the assumptions log (assumption A-35) and listed with the others in section 9.

**The measures, in words.**

| Measure | How it is counted |
|---|---|
| Joins, 12 steps | Inside graphs 1–5, 78–82 and 187–191 there are 12 steps from one graph to the next. A **join** is a sensor that is in the red four of a window and was not in the red four of the window before. This counts the joins over those 12 steps. |
| Step ÷ size, 12 steps | On the same fifteen graphs, and on **sizes**, each Shapley value with its sign dropped: for one sensor, the average change in its size over the 12 steps, divided by its average size over the fifteen graphs. Then the median over the 14 sensors. 0.1 would mean a sensor's size typically moves by a tenth of itself from one graph to the next. |
| Joins per step | The joins over every pair of neighbouring windows of the engine, divided by the number of pairs. |
| Steps with a join | The share of those pairs at which at least one sensor joins. 0.9 would mean the red four differs from the window before at nine steps in ten. |
| Steps with two or more joins | The share of those pairs at which two, three or four sensors join, so that at least half of the red four is new. |
| Step ÷ size, every step | Step ÷ size over every pair of neighbouring windows, again on sizes, with the average size taken over every window. |
| Stage jitter | The same idea one stage earlier, for the number the temporal stage hands the graph layers: per sensor, its average change from one window to the next divided by its spread (standard deviation) over the engine's windows, then the median over the sensors. |

The first two are the ones Table 12 can be counted on as well, because it holds only the fifteen graphs.

**Why sizes.** Table 12 holds positive numbers only (finding F-08), so it cannot show a value changing sign, and the red four is chosen by size anyway. A made-up example: a value going from +2 to −2 has not changed size, so it adds 0 to the step on sizes; on the signed values it would add a step of 4.

A made-up example of a join: if one window's red four is {Nc, T24, W31, T30} and the next window's is {Nc, T24, W31, T50}, T50 has joined and T30 has left, which is one join. One step can have at most four.

**Which engine.** For each seed, the held-back engine with the most windows: the network never learned from it, and it is long enough to reach graph 191. The seed decides which engines are held back, so the two runs hold back the same engines for the same seed; a cell below checks that they explain the same one.

The next cell does the work: ten networks, every window of five engines, 16,384 groups of sensors per window. `shap_for_each_seed` loads one saved network at a time, refuses it if it does not score what its run recorded on the test engines, and returns `dynamic_shap`'s table for the engine. It takes several minutes on a processor.

In [ ]:
def red_four(values) -> set[str]:
    """The four sensors whose values are largest in size, ties kept in sensor order."""
    order = sorted(range(len(values)), key=lambda i: -abs(values[i]))
    return {sensors[i] for i in order[:4]}


def joins_at_each_step(rows, pairs) -> np.ndarray:
    """How many sensors enter the red four at each of the given (before, after) pairs of rows."""
    return np.array([len(red_four(rows[after]) - red_four(rows[before])) for before, after in pairs])


def joins(rows, pairs) -> int:
    """How many times a sensor enters the red four, over the given (before, after) pairs of rows."""
    return int(joins_at_each_step(rows, pairs).sum())


def step_over_size(rows, pairs) -> float:
    """Per sensor, the average step in size between paired rows over the average size; median over sensors."""
    sizes = np.abs(rows)
    steps = np.abs(np.array([sizes[after] - sizes[before] for before, after in pairs]))
    return float(np.median(steps.mean(axis=0) / sizes.mean(axis=0)))


def stage_jitter(handed) -> float:
    """Per sensor, the average step of what the stage hands over, over its spread; median over sensors."""
    return float(np.median(np.abs(np.diff(handed, axis=0)).mean(axis=0) / handed.std(axis=0)))


# The 12 steps inside Table 12's three stages, as pairs of its fifteen columns.
TABLE_STEPS = [(k, k + 1) for start in (0, 5, 10) for k in range(start, start + 4)]
# ASSUMPTION A-35: graph g of the paper's Table 12 and Fig. 10 is the engine's g-th window,
# ending at cycle g + 29; the paper never says which cycle a graph is. So the fifteen graphs
# are these rows of an engine's table.
DRAWN = [graph - 1 for graph in SHAP_TABLE_12_GRAPHS]

measured, followed = [], {}
started = time.perf_counter()
for stage, stored in stage_runs.items():
    # shap_for_each_seed loads one saved network at a time, checks that it scores
    # what the run recorded on the test engines, and explains every window of the
    # held-back engine with the most windows.
    for explained in shap_for_each_seed(stored):
        (engine_unit, engine_table), = explained.tables.items()
        held = explained.prepared.validation
        rows = np.flatnonzero(held.units == engine_unit)
        rows = rows[np.argsort(held.end_cycles[rows])]
        with torch.no_grad():
            handed = explained.model.temporal(torch.from_numpy(held.features[rows])).numpy()
        assert handed.shape[2] == 1, "both stages hand the graph layers one number per sensor"
        handed = handed[:, :, 0]
        explained_values = engine_table[sensors].to_numpy()
        assert len(explained_values) >= SHAP_TABLE_12_GRAPHS[-1], "the engine must reach graph 191"
        drawn = explained_values[DRAWN]
        neighbours = [(t - 1, t) for t in range(1, len(explained_values))]
        each_step = joins_at_each_step(explained_values, neighbours)
        measured.append({
            "stage": stage,
            "seed": explained.seed,
            "engine": engine_unit,
            "windows": len(explained_values),
            "held back": tuple(int(u) for u in np.unique(held.units)),
            "joins, 12 steps": joins(drawn, TABLE_STEPS),
            "step ÷ size, 12 steps": step_over_size(drawn, TABLE_STEPS),
            "joins per step": float(each_step.mean()),
            "steps with a join": float((each_step >= 1).mean()),
            "steps with two or more joins": float((each_step >= 2).mean()),
            "median joins at a step": float(np.median(each_step)),
            "step ÷ size, every step": step_over_size(explained_values, neighbours),
            "stage jitter": stage_jitter(handed),
            "largest miss (cycles)": float(np.abs(
                explained_values.sum(axis=1) - (engine_table["predicted"] - engine_table["baseline"]).to_numpy()).max()),
        })
        if explained.seed == SEED:
            followed[stage] = {"unit": engine_unit, "table": engine_table, "handed": handed}
        print(f"{STAGE_NAMES[stage]:18s} seed {explained.seed}: engine {engine_unit:3d}, {len(explained_values)} windows "
              f"({time.perf_counter() - started:5.0f} s so far)")

measured = pd.DataFrame(measured)

**Check.** Every one of these explanations must add up, window by window, as in step 5. Before that check, the cell lists the red four of the first four windows of seed 42's default network and what joined at each step, so that the counts below can be followed by eye.

In [ ]:
default_table = followed["last_cycle"]["table"]
first_rows = default_table[sensors].to_numpy()[:4]
print(f"Seed {SEED}'s default network, engine {followed['last_cycle']['unit']}, its first four windows:\n")
before = None
for row, values in zip(default_table.itertuples(), first_rows):
    four = red_four(values)
    named = ", ".join(f"{s} {values[sensors.index(s)]:+.2f}" for s in sorted(four, key=lambda s: -abs(values[sensors.index(s)])))
    entered = sorted(four - before) if before is not None else []
    print(f"  cycle {row.end_cycle}: {named}"
          + (f"   joins: {', '.join(entered) or 'none'} ({len(entered)})" if before is not None else ""))
    before = four

step8_adds_up = bool(measured["largest miss (cycles)"].max() < 1e-4)
assert step8_adds_up
print(f"\nOK  All {measured['windows'].sum():,} explained windows add up, the largest miss "
      f"{measured['largest miss (cycles)'].max():.1e} cycles.")

### The five seeds, and the paper counted the same way

The first table holds every seed of both runs. The second gives the mean and the spread (standard deviation) over the five seeds, and adds the paper's Table 12, counted with the same functions. Table 12 holds only its fifteen graphs, so only the two measures over the 12 steps can be counted on it.

In [ ]:
COLUMNS = ["joins, 12 steps", "step ÷ size, 12 steps", "joins per step", "steps with a join",
           "steps with two or more joins", "step ÷ size, every step", "stage jitter"]
print("Every seed of both runs:")
display(measured[["stage", "seed", "engine", "windows", *COLUMNS, "median joins at a step"]].round(3))

paper = np.array([[SHAP_TABLE_12[s][column] for s in sensors] for column in range(len(SHAP_TABLE_12_GRAPHS))])
paper_joins = joins(paper, TABLE_STEPS)
paper_step_over_size = step_over_size(paper, TABLE_STEPS)
entered = [(SHAP_TABLE_12_GRAPHS[b], sorted(red_four(paper[b]) - red_four(paper[a]))) for a, b in TABLE_STEPS
           if red_four(paper[b]) - red_four(paper[a])]


def mean_and_spread(places):
    """Format a column of five seeds as its mean and standard deviation."""
    return lambda c: f"{c.mean():.{places}f} ± {c.std(ddof=1):.{places}f}"


summary = pd.DataFrame({
    column: measured.groupby("stage")[column].agg(mean_and_spread(1 if column == COLUMNS[0] else 3))
    for column in COLUMNS
})
summary.loc["the paper's Table 12"] = [str(paper_joins), f"{paper_step_over_size:.3f}", *["—"] * (len(COLUMNS) - 2)]
summary = summary.loc[["the paper's Table 12", "lstm", "last_cycle"]]
summary.index = ["the paper's Table 12", "recurrent network (lstm)", "default network (last_cycle)"]
print("Mean ± spread (standard deviation) over the five seeds:")
display(summary)
print(f"In Table 12 the one sensor to enter the four largest: {entered}")

### Is the difference larger than the seed-to-seed spread?

The same seed holds back the same engines in both runs, and both explain the same engine, so the two can be compared seed by seed: five differences, default minus recurrent. A **paired t-test** asks whether those five differences sit far enough from zero, given how much they vary among themselves, that seed-to-seed chance is an unlikely explanation. Its `t` is the average difference divided by its standard error, which is the spread of the differences divided by the square root of their number. Its `p` is the probability of a `t` at least that far from zero if the two stages did not differ on average; with five differences there are 4 degrees of freedom, one fewer than the number of differences.

A made-up example: differences of 2, 3 and 4 have mean 3 and spread 1, so the standard error is 1 / √3 = 0.577 and `t` = 3 / 0.577 = 5.2.

In [ ]:
by_seed = measured.pivot(index="seed", columns="stage")
same_engines = bool((by_seed["held back"]["last_cycle"] == by_seed["held back"]["lstm"]).all()
                    and (by_seed["engine"]["last_cycle"] == by_seed["engine"]["lstm"]).all())
assert same_engines
print(f"OK  For every seed both runs held back the same {len(by_seed['held back']['lstm'].iloc[0])} engines "
      "and followed the same one: "
      f"{by_seed['engine']['lstm'].to_dict()}\n")

tests = []
for column in COLUMNS:
    default, recurrent = by_seed[column]["last_cycle"], by_seed[column]["lstm"]
    difference = default - recurrent
    result = stats.ttest_rel(default, recurrent)
    tests.append({
        "measure": column,
        "default minus recurrent, mean": round(float(difference.mean()), 3),
        "spread of the five differences": round(float(difference.std(ddof=1)), 3),
        "t": round(float(result.statistic), 2),
        "p": f"{result.pvalue:.1e}",
        "default higher on every seed": bool((difference > 0).all()),
    })
tests = pd.DataFrame(tests).set_index("measure")
display(tests)
default_joins_more = bool(tests.loc[["joins, 12 steps", "joins per step"],
                                    "default higher on every seed"].all())

### The paper's Fig. 10 beside seed 42's engine

Each small graph is drawn the way the playground draws one, and is one window. The fourteen sensors sit round a circle in the order the paper's figure places them (`FIG_10_RING`), joined by the connections of the paper's Table 3. The red four are red, darker the larger their value, and each carries its name on its own node; the rest are blue, fading with rank. As in the playground, each stage is a block of rows. The top row of a block is the paper's, redrawn from the numbers of its Table 12 (`SHAP_TABLE_12`). The printed figure has the same red four in 13 of the 15 graphs; in graph 81 it colours T30 red in place of phi, and in graph 82 T30 and Nc in place of P30 and phi (finding F-17). The two rows below it are this engine at the same graph numbers, under the default network and under the recurrent one, both seed 42; each title names the graph and the cycle its window ends at.

In [ ]:
RING = list(FIG_10_RING)
# The playground's colours for Fig. 10: red for the four largest, blue for the rest.
BURN, FAN, EDGE, INK, PAPER = "#b3261e", "#2f7fd1", "#c3cad4", "#16191d", "#ffffff"
# The playground's proportions: a node's radius is 0.21 of the ring's, and a name
# sits on its own node, drawn over a white outline after every node.
NODE = 0.21
HALO = [patheffects.withStroke(linewidth=1.8, foreground=PAPER)]
raw_graph, graph_symbols = build_physics_adjacency()
EDGES = [(graph_symbols[a], graph_symbols[b]) for a in range(len(graph_symbols))
         for b in range(a + 1, len(graph_symbols)) if raw_graph[a, b] or raw_graph[b, a]]
# Round a circle in Fig. 10's order, clockwise from the node just left of the top.
angles = np.deg2rad(-90 - 180 / len(RING) + np.arange(len(RING)) * 360 / len(RING))
AT = {s: (np.cos(a), -np.sin(a)) for s, a in zip(RING, angles)}


def ring(axis, values, title):
    """One small graph of Fig. 10: the sensors round a circle, Table 3's edges, the red four named on their nodes."""
    axis.set_aspect("equal")
    axis.axis("off")
    for a, b in EDGES:
        axis.plot([AT[a][0], AT[b][0]], [AT[a][1], AT[b][1]], color=EDGE, linewidth=0.5, zorder=1)
    size = {s: abs(values[s]) for s in sensors}
    ranked = sorted(sensors, key=lambda s: -size[s])
    top = size[ranked[0]] or 1.0
    for rank, s in enumerate(ranked):
        colour, alpha = ((BURN, 0.45 + 0.55 * size[s] / top) if rank < 4
                         else (FAN, 1 - (rank - 4) / 10 * 0.8))
        axis.add_patch(Circle(AT[s], NODE, facecolor=to_rgba(colour, alpha), edgecolor=INK, linewidth=0.3, zorder=2))
    for s in ranked[:4]:
        axis.text(*AT[s], s, ha="center", va="center", fontsize=6.5, fontweight="bold", family="monospace",
                  color=INK, path_effects=HALO, zorder=3)
    axis.set_xlim(-1.25, 1.25)
    axis.set_ylim(-1.25, 1.25)
    axis.set_title(title, fontsize=8, pad=2)


followed_unit = followed["last_cycle"]["unit"]
rows_drawn = [("The paper,\nTable 12", None)] + [
    (f"{STAGE_NAMES[stage].capitalize()},\nseed {SEED}, engine {followed_unit}", stage) for stage in ("last_cycle", "lstm")]
STAGES = [("Early: graphs 1 to 5", 0), ("Middle: graphs 78 to 82", 5), ("Late: graphs 187 to 191", 10)]
# As in the playground, each stage is a block: the paper's five graphs above this
# engine's at the same graph numbers. A thin empty row separates the blocks.
figure = plt.figure(figsize=(9.5, 16))
grid = figure.add_gridspec(11, 5, height_ratios=[1, 1, 1, 0.3] * 2 + [1, 1, 1],
                           left=0.2, right=0.99, top=0.965, bottom=0.005, wspace=0.08, hspace=0.3)
for block, (heading, first) in enumerate(STAGES):
    for row, (label, stage) in enumerate(rows_drawn):
        for place in range(5):
            column, axis = first + place, figure.add_subplot(grid[4 * block + row, place])
            graph = SHAP_TABLE_12_GRAPHS[column]
            if stage is None:
                values, title = dict(zip(sensors, paper[column])), f"graph {graph}"
            else:
                shown = followed[stage]["table"].iloc[graph - 1]
                values, title = {s: shown[s] for s in sensors}, f"graph {graph}, cycle {int(shown['end_cycle'])}"
            ring(axis, values, title)
            if place == 0:
                axis.text(-1.45, 0, label, ha="right", va="center", fontsize=9)
            if row == 0 and place == 2:
                axis.annotate(heading, xy=(0.5, 1.22), xycoords="axes fraction", ha="center", fontsize=11,
                              fontweight="bold")
plt.show()

counts = {"the paper": paper_joins}
for stage in ("last_cycle", "lstm"):
    drawn_rows = followed[stage]["table"][sensors].to_numpy()[DRAWN]
    counts[f"{STAGE_NAMES[stage]}, seed {SEED}"] = joins(drawn_rows, TABLE_STEPS)
print("Joins over the 12 steps drawn: " + "; ".join(f"{who} {n}" for who, n in counts.items()))

### Why the default network's four change so often

A sensor's Shapley value measures how the prediction moves when that sensor's input is swapped for its stand-in. What the graph layers receive of that input is one number per sensor, and the two stages make it differently:

- The default network passes on the window's **last** reading. Neighbouring windows share 29 of their 30 cycles, but the last cycle is a new one every time, and one reading still carries whatever noise the smoothing left in it.
- The recurrent network computes its number from **all 30** readings, 29 of which the previous window also had.

The cell below shows the number each stage hands over for one sensor, T50, on the first five windows of the engine, and the stage jitter for T50 alone.

In [ ]:
t50 = sensors.index("T50")
first_five = followed["last_cycle"]["table"]["end_cycle"].iloc[:5].tolist()
print(f"What each stage hands the graph layers for T50, seed {SEED}, engine {followed_unit}, "
      f"windows ending at cycles {first_five[0]} to {first_five[-1]}:\n")
for stage in ("last_cycle", "lstm"):
    handed = followed[stage]["handed"][:5, t50]
    print(f"  {STAGE_NAMES[stage]:18s} " + "  ".join(f"{v:+.3f}" for v in handed))
for stage in ("last_cycle", "lstm"):
    handed = followed[stage]["handed"][:, t50]
    print(f"\n{STAGE_NAMES[stage]}: T50's average step {np.abs(np.diff(handed)).mean():.3f}, "
          f"its spread over the engine's life {handed.std():.3f}, "
          f"step over spread {np.abs(np.diff(handed)).mean() / handed.std():.3f}")

#### Reading the result

**The default network's red four changes at almost every window.** Over every pair of neighbouring windows of the engine, at least one sensor joins the red four at 0.917 ± 0.027 of the steps on the default network and at 0.364 ± 0.093 on the recurrent one. Two or more join at 0.453 ± 0.060 of the default network's steps and at 0.021 ± 0.017 of the recurrent network's. The median step of the default network has one join on seeds 42, 43 and 44 and two on seeds 45 and 46 (the last column of the first table). On average 1.469 ± 0.119 sensors join per step on the default network, against 0.384 ± 0.110. Inside the 12 steps the paper's figure draws, that is 20.2 ± 4.8 joins against 4.6 ± 2.1, where the paper's Table 12 has 1; every seed of both networks has more than that one.

**The sizes move more on the default network as well.** Over those 12 steps a typical sensor's size changes by 0.394 ± 0.107 of itself on the default network and by 0.152 ± 0.059 on the recurrent one; over every step, by 0.365 ± 0.081 and 0.134 ± 0.035. Table 12's ratio over its 12 steps is 0.084. The paired test above compares the two networks only. Whether the recurrent network's sizes move more than Table 12's is not tested, and the seeds do not all say so: on seed 42 the recurrent network's ratio over the 12 steps, 0.078, is below Table 12's.

**The difference is far larger than the seed-to-seed spread.** Paired by seed, the default network has 15.6 more joins in the 12 steps (p = 4.9e-04), a join at 0.553 more of its steps (p = 5.3e-05), two or more joins at 0.433 more of its steps (p = 2.9e-05) and 1.085 more joins per step (p = 2.0e-07), and it is the higher of the two on every one of the five seeds, on all seven measures.

**It goes with what the stage hands over.** The stage jitter is 0.281 ± 0.021 for the default network and 0.119 ± 0.005 for the recurrent one. On the first five windows of engine 5, T50's number from the default network jumps between −1.020 and −1.515, while the recurrent network's stays between 0.756 and 0.814. This is measured beside the joins, not shown to cause them: the two runs differ in the whole network trained on each stage's numbers, not only in the numbers.

**What it does not settle.** Even the recurrent network's red four changes more often than Table 12's: 4.6 ± 2.1 joins in the 12 steps against 1, and more than 1 on every seed. Table 12 holds positive magnitudes computed in a way the paper never states (finding F-08), so that remaining gap cannot be put down to the network. And seed 42, the seed this notebook follows and the explanations stored beside the recurrent run use, gave the fewest recurrent joins of the five, 2: its picture is the calmest of the five, not a typical one.

**Check.** Four statements above rest on counts that must hold: Table 12, counted with this notebook's functions, has 1 join and a step-to-size ratio of 0.084; every seed of both networks has more joins in the 12 steps than Table 12; the recurrent network's step-to-size ratio over the 12 steps is below Table 12's on seed 42 and on no other seed; and the default network has more joins than the recurrent one on every seed, inside the 12 steps and per step.


In [ ]:
table12_recount = paper_joins == 1 and round(paper_step_over_size, 3) == 0.084
assert table12_recount
assert default_joins_more
more_than_table12 = bool((measured["joins, 12 steps"] > paper_joins).all())
assert more_than_table12
recurrent = measured[measured["stage"] == "lstm"]
calmer_than_table12 = recurrent.loc[recurrent["step ÷ size, 12 steps"] < paper_step_over_size, "seed"].tolist()
assert calmer_than_table12 == [SEED]
print("OK  Table 12, counted the same way: 1 join in its 12 steps, step over size 0.084.")
print("OK  On every seed the default network has more joins than the recurrent one, "
      "both inside the 12 steps and per step over the whole engine.")
print(f"OK  Every seed of both networks has more joins in the 12 steps than Table 12's {paper_joins}.")
print(f"OK  The recurrent network's step over size in the 12 steps is below Table 12's "
      f"{paper_step_over_size:.3f} on seed {calmer_than_table12[0]} only.")


## 7. What a Shapley value does and does not show

1. **What object is explained?** One prediction: one window in, one number of cycles out. Step 5's picture is many such explanations side by side, not an explanation of the network as a whole.
2. **What does the method have access to?** Only the network's inputs and outputs. It never reads a parameter or a gradient; it asks the network for 16,384 predictions per window. So it could explain any model that takes these inputs. This project's code takes one shortcut that reads how the network is built: it runs the temporal stage once per window instead of once per prediction, which gives the same numbers, and refuses a network for which it would not (`explain/README.md`, "Why the temporal stage runs only once per window").
3. **What does it change?** Whole sensors. A sensor's 30 readings are kept or replaced together, never cycle by cycle, so it cannot say which cycles inside the window mattered. It never changes the sensor graph either.
4. **What does it measure?** How far the prediction moves when a sensor joins a group of the others, averaged over every group with the weights of Equation 15.
5. **What is the result?** Fourteen signed numbers per window, in cycles, adding up to the prediction minus the reference prediction.
6. **What does it assume?**
   - That "absent" is best represented by the sensor's average training window (assumption A-20). A different stand-in gives different values.
   - That the network's answers for windows mixing real and average readings mean something, although no real engine ever produced such a window.
   - That credit for an interaction between sensors should be split evenly among them.
7. **What does it not show?**
   - **Cause.** `Nc` having the largest contributions near failure says this network's prediction moved most with `Nc`'s readings. It does not say the core speed causes wear, or that it is physically the best sign of wear.
   - **That the prediction was right.** A wrong prediction is split up just as neatly, and the values add up even for a network that was never trained, because efficiency is a property of the arithmetic.
   - **Which connections carried the information.** Credit goes to sensors, not to the edges of the sensor graph between them. Which edges a prediction depended on is a different question, and the one this thesis goes on to ask.
   - **Anything about another design.** Step 7 compares five networks of one design. Step 8 shows that a network with the paper's recurrent stage explains differently: its four largest contributions change far less often from one window to the next. A different sensor graph would need its own explanation.

**Do not confuse** "`Nc` has the largest Shapley values near failure" with "`Nc` is the most important measurement of engine wear". The first is measured above. The second would need evidence about engines, not about one network.

## 8. Final verification

Every check of this notebook, in one place. A reader who trusts nothing else should be able to run this cell.

In [ ]:
checks = [
    ("The loaded network scores what the run recorded", same_network, f"seed {SEED}"),
    ("The reference prediction is a possible prediction", reference_in_range, f"{reference:.2f} cycles"),
    ("Three sensors: the values add up", three_add_up, "step 3"),
    ("Three sensors: orders and groups agree", routes_agree, "Equation 15's weights"),
    ("Fourteen sensors: the values add up", fourteen_add_up, f"{window_error:.1e} cycles"),
    (f"Every window of engine {engine} adds up", every_window_adds_up,
     f"{engine_gap:.1e} cycles over {len(attribution)} windows"),
    ("The label column is the remaining life capped at 125", label_is_capped_life,
     f"below the life truly left in {below_true_life} of {len(attribution)} windows"),
    ("P15 is ranked last for this engine", p15_last, f"rank {p15_rank}"),
    ("P15 is ranked last by all five networks", p15_last_everywhere, "100 test engines"),
    ("Nc and T24 lead for all five networks", top_two_everywhere, "100 test engines"),
    ("Step 8's two runs differ only in the temporal stage", same_settings, stage_runs['lstm'].summary['commit']),
    ("Both runs explain the same engine for each seed", same_engines, "5 seeds"),
    ("Step 8's explanations all add up", step8_adds_up,
     f"{measured['largest miss (cycles)'].max():.1e} cycles over {measured['windows'].sum():,} windows"),
    ("Table 12 recounted: 1 join, step / size 0.084", table12_recount, "12 steps"),
    ("The default network joins more on every seed", default_joins_more, "both counts"),
    ("Every seed joins more often than Table 12", more_than_table12, "12 steps"),
    ("Only seed 42's recurrent sizes move less than Table 12's", calmer_than_table12 == [SEED], "12 steps"),
    ("The digests differ only by the stored description", digest_is_the_description, "ba82e3"),
]
width = max(len(name) for name, _, _ in checks) + 2
print(f"{'CHECK':<{width}} RESULT   DETAIL")
print("-" * (width + 40))
for name, passed, detail in checks:
    print(f"{name:<{width}} {'PASS' if passed else 'FAIL':<8} {detail}")
print("-" * (width + 40))
failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise AssertionError(f"{len(failed)} check(s) failed: {failed}")
print(f"All {len(checks)} checks passed.")

## 9. Summary

### What this notebook produced

- **Seed 42's trained network, loaded from a stored run** and confirmed to be that run's network by its test error.
- **Shapley values worked by hand** for three sensors, over every order they could arrive in, and shown to equal the group-by-group form of Equation 15.
- **Exact Shapley values for all fourteen sensors**, for one window and then for every window of one held-back engine's life, adding up to the prediction minus the reference prediction at every window.
- **A chart of how each sensor's contribution moved over that engine's life**, and a table of the same by stage: many small, opposing contributions while the engine is healthy, reorganising around `Nc` as it nears failure.
- **A ranking that holds across all five of the run's networks**, with `P15` last and `Nc` and `T24` first.
- **A count of how often the four largest contributions change** from one window to the next, on five seeds of two runs alike but for the temporal stage, beside the paper's Fig. 10 redrawn from its Table 12: the default network's four change at almost every window, the recurrent network's far less often, and Table 12's fewer times than on any seed of either.

### Decisions that were not read from the paper

| Decision | Value used | Written up as |
|---|---|---|
| What stands in for a removed sensor | Its average window over the training engines | A-20 |
| What the fourteen values add up to | The prediction minus the prediction for the stand-in window, not the average label | A-24 |
| Which engine is followed over its life | The held-back engine with the most windows | this notebook |
| How stages of life are summarised | Thirds of the engine's windows | this notebook |
| What counts as a change of the four largest | A sensor entering the four largest by size from one window to the next | this notebook |
| Which engine step 8 follows for each seed | The held-back engine with the most windows, the same in both runs | this notebook |
| Which window graph *g* of Fig. 10 and Table 12 is | The engine's *g*-th window, the one ending at cycle *g* + 29 | A-35 |
| How far a value moves from one window to the next | Measured on sizes, the sign dropped, because Table 12 holds sizes only | this notebook |

### What comes next

- **`02_occlusion_fd001.ipynb`** explains the same network with a cheaper method that is not in the paper, and shows where it agrees with the Shapley values and where it cannot.
- **Edges instead of sensors.** Every value here credits a sensor. Which connections of the sensor graph a prediction depended on is the question the paper does not ask and this thesis does.

### References

- Shapley, L. S. (1953). A value for n-person games. *Contributions to the Theory of Games II*, 307-317.
- Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS 30*.
- Wen, J.-R., et al. (2026). Explainable spatiotemporal graph neutral network with dynamic SHAP enhancement for aeroengine RUL prediction. *Aerospace Science and Technology*, 168, 110875.

Full citations are in `docs/REFERENCES.md`.